# KahnemanGuard: System-1 ModernBERT-large on Google Colab NVIDIA L4 GPU

**Đề tài:** Nghiên cứu phương pháp phân tầng chọn lọc định hướng rủi ro nhằm kiểm duyệt hiệu quả đầu vào của mô hình ngôn ngữ lớn

**Mục tiêu Notebook:**
1. Huấn luyện System 1 (`ModernBERT-large` 395M) trên GPU **NVIDIA L4 (24GB VRAM - Ada Lovelace)** sử dụng **FlashAttention-2** & **Bfloat16**.
2. Hỗ trợ 2 chế độ: **Baseline (Cross-Entropy)** và **RLCD (Reinforcement Learning from Calibration Drift)**.
3. Đo lường định lượng **ECE (Expected Calibration Error)**, **Brier Score**, lập biểu đồ **Reliability Diagram**.
4. Lượng tử hóa sang **ONNX INT8** và kiểm thử độ trễ suy luận CPU (cam kết ~12ms).
5. Tự động lưu Checkpoint và Báo cáo vào Google Drive hoặc đồng bộ qua VS Code Remote-SSH.

In [2]:
# ======================================================================
# BƯỚC 1: XÁC THỰC MÔI TRƯỜNG PHẦN CỨNG GPU L4 & BFLOAT16
# ======================================================================
import os

import torch

print("=== KIỂM TRA PHẦN CỨNG COLAB ===")
assert torch.cuda.is_available(), "Cảnh báo: Chưa kích hoạt GPU! Vào Runtime -> Change runtime type -> Chọn L4 GPU."
gpu_name = torch.cuda.get_device_name(0)
total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"* GPU Device:       {gpu_name}")
print(f"* VRAM Capacity:    {total_mem:.2f} GB (NVIDIA Ada Lovelace)")
print(f"* BF16 Supported:   {torch.cuda.is_bf16_supported()}")
print(f"* PyTorch Version:  {torch.__version__}")

!nvidia-smi

=== KIỂM TRA PHẦN CỨNG COLAB ===
* GPU Device:       NVIDIA L4
* VRAM Capacity:    23.66 GB (NVIDIA Ada Lovelace)
* BF16 Supported:   True
* PyTorch Version:  2.11.0+cu130
Tue Nov 10 06:19:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   40C    P8             16W /   72W |       3MiB /  23

In [10]:
# ======================================================================
# BƯỚC 2: CÀI ĐẶT THƯ VIỆN CHO NVIDIA L4 (ADA LOVELACE)
# ======================================================================
import subprocess
import sys

pkgs = [
    'transformers>=4.48.0',
    'datasets>=3.0.0',
    'accelerate',
    'safetensors',
    'scikit-learn',
    'tabulate',
    'onnx',
    'onnxruntime',
    'matplotlib',
]
print('Đang cài đặt các thư viện cần thiết...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', *pkgs])
print('Cài đặt hoàn tất 100%!')


Đang cài đặt các thư viện cần thiết...
Cài đặt hoàn tất 100%!


In [ ]:
# ======================================================================
# BƯỚC 3: CLONE MÃ NGUỒN & BENCHMARKS KG-EVAL TỪ GITHUB REPOSITORY
# ======================================================================
import os
import sys
import subprocess

TARGET_DIR = "/content/kahneman-guard"
REPO_URL = "https://github.com/TheWilsonGlobal/kahneman-guard-pub.git"

print(f"--> [1/2] Cloning mã nguồn và benchmarks từ {REPO_URL}...")
if os.path.exists(TARGET_DIR):
    subprocess.run(["rm", "-rf", TARGET_DIR], check=True)

subprocess.check_call(["git", "clone", REPO_URL, TARGET_DIR])
print(f"--> Đã clone thành công vào: {TARGET_DIR}")

# 2. Xác thực tính toàn vẹn của bộ benchmark KG-Eval v1.0 và script huấn luyện
print("--> [2/2] Xác thực bộ dữ liệu kiểm định độc lập và mã huấn luyện...")
BENCHMARK_DIR = os.path.join(TARGET_DIR, "data/benchmarks")
TRAIN_SCRIPT = os.path.join(TARGET_DIR, "notebooks/colab_L4/train_l4.py")

assert os.path.exists(TRAIN_SCRIPT), f"Lỗi: Thiếu {TRAIN_SCRIPT}"
assert os.path.exists(os.path.join(BENCHMARK_DIR, "kg_eval_cal.jsonl")), "Lỗi: Thiếu kg_eval_cal.jsonl"
assert os.path.exists(os.path.join(BENCHMARK_DIR, "kg_eval_route.jsonl")), "Lỗi: Thiếu kg_eval_route.jsonl"
assert os.path.exists(os.path.join(BENCHMARK_DIR, "kg_eval_test.jsonl")), "Lỗi: Thiếu kg_eval_test.jsonl"
assert os.path.exists(os.path.join(BENCHMARK_DIR, "kg_eval_frozen_checksums.sha256")), "Lỗi: Thiếu file checksum SHA-256"

print(" HOÀN TẤT 100%! Toàn bộ dữ liệu kiểm định KG-Eval và mã nguồn đã sẵn sàng trên Colab L4!")


In [ ]:
# ======================================================================
# BƯỚC 4: TẢI & TOKENIZE FT-DATA (PROMPT-ONLY ALIGNED) CHO MODERNBERT
# ======================================================================
import random

from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_ID = "answerdotai/ModernBERT-large"
MAX_SAMPLES = 25000
MAX_LEN = 512
DATA_PATH = "/content/train_items.pt"

tok = AutoTokenizer.from_pretrained(MODEL_ID)
raw_samples = []

print("--> [1/2] Nạp ToxicChat (Prompt Toxicity & Jailbreak)...")
try:
    ds_toxic = load_dataset("lmsys/toxic-chat", "toxicchat0124", split="train")
    for row in ds_toxic:
        p = row["user_input"].strip()
        lbl = int((row.get("toxicity", 0) == 1) or (row.get("jailbreaking", 0) == 1))
        raw_samples.append((p, lbl))
except Exception as e:  # noqa: BLE001
    print(f"Warning ToxicChat: {e}")

print("--> [2/2] Nạp BeaverTails (Prompt-Only Aligned)...")
try:
    ds_beaver = load_dataset("PKU-Alignment/BeaverTails", split="30k_train")
    for row in ds_beaver:
        p = row["prompt"].strip()
        lbl = int(not row["is_safe"])
        raw_samples.append((p, lbl))
except Exception as e:  # noqa: BLE001
    print(f"Warning BeaverTails: {e}")

valid_samples = [(p, lbl) for p, lbl in raw_samples if len(p) > 5]
random.seed(42)
random.shuffle(valid_samples)
valid_samples = valid_samples[:MAX_SAMPLES]

train_items = []
for prompt, label in valid_samples:
    enc = tok(prompt, truncation=True, max_length=MAX_LEN)
    target_dist = [0.05, 0.95] if label == 1 else [0.95, 0.05]
    train_items.append({
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "target": target_dist,
        "label": label
    })

torch.save(train_items, DATA_PATH)
print(f"--> Đã tokenize và lưu cache {len(train_items)} mẫu vào {DATA_PATH}")

In [ ]:
# ======================================================================
# BƯỚC 5: KHỞI CHẠY HUẤN LUYỆN TRÊN NVIDIA L4 (train_l4.py)
# ======================================================================
import os
import sys

DATA_PATH = '/content/train_items.pt'
SCRIPT_PATH = '/content/kahneman-guard/notebooks/colab_L4/train_l4.py'
OUTPUT_CE = '/content/model_ce_baseline'
OUTPUT_RLCD = '/content/model_rlcd_calibrated'

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError("Lỗi: Chưa tìm thấy file /content/train_items.pt! Vui lòng bấm chạy Cell 4 ở trên.")

print('=== [1/2] BẮT ĐẦU HUẤN LUYỆN MODEL 1 BASELINE (CROSS-ENTROPY) ===')
!python {SCRIPT_PATH} --data_path {DATA_PATH} --output_dir {OUTPUT_CE} --mode ce --epochs 3 --batch_size 16 --grad_accum 2

print('\n=== [2/2] BẮT ĐẦU HUẤN LUYỆN MODEL 1 RLCD CALIBRATED ===')
!python {SCRIPT_PATH} --data_path {DATA_PATH} --output_dir {OUTPUT_RLCD} --mode rlcd --epochs 3 --batch_size 16 --grad_accum 2


In [ ]:
# ======================================================================
# BƯỚC 6: ĐO ĐẠC SAI SỐ HIỆU CHUẨN ECE, BRIER SCORE & BIỂU ĐỒ RELIABILITY
# ======================================================================
import os
import sys
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from tabulate import tabulate
from transformers import AutoTokenizer
from safetensors.torch import load_file

if '/content/kahneman-guard/notebooks/colab_L4' not in sys.path:
    sys.path.insert(0, '/content/kahneman-guard/notebooks/colab_L4')
from train_l4 import KahnemanDecisionGuard

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model_id = 'answerdotai/ModernBERT-large'
tok = AutoTokenizer.from_pretrained(model_id)

# 1. Nạp tập kiểm định độc lập kg_eval_cal.jsonl (300 prompts)
eval_path = '/content/kahneman-guard/data/benchmarks/kg_eval_cal.jsonl'
prompts, true_labels = [], []
with open(eval_path, 'r', encoding='utf-8') as f:
    for line in f:
        d = json.loads(line)
        prompts.append(d['prompt'])
        true_labels.append(1 if d['is_malicious'] else 0)
true_labels = np.array(true_labels)
print(f'--> Đã nạp {len(prompts)} mẫu độc lập từ {eval_path}')

# 2. Hàm suy luận trích xuất xác suất dự đoán đồng nhất BFloat16
def get_predictions(model_dir, apply_temp=True):
    model = KahnemanDecisionGuard(model_id, use_flash_attn=False)
    weights = load_file(os.path.join(model_dir, 'model.safetensors'))
    model.load_state_dict(weights)
    model.to(device=device, dtype=dtype)
    model.eval()
    
    temp = 1.0
    if apply_temp and os.path.exists(os.path.join(model_dir, 'guardrail_config.json')):
        with open(os.path.join(model_dir, 'guardrail_config.json')) as f:
            temp = json.load(f).get('temperature', 1.0)
            
    all_probs = []
    bs = 32
    with torch.no_grad():
        for i in range(0, len(prompts), bs):
            batch_prompts = prompts[i:i+bs]
            enc = tok(batch_prompts, padding=True, truncation=True, max_length=512, return_tensors='pt').to(device)
            with torch.autocast('cuda', dtype=dtype):
                logits = model(enc['input_ids'], enc['attention_mask'])
            scaled_logits = logits.float() / temp
            probs = torch.softmax(scaled_logits, dim=-1).cpu().numpy()
            all_probs.extend(probs)
    return np.array(all_probs), temp

# 3. Tính toán ECE và Brier Score
def compute_metrics(probs, labels, n_bins=10):
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies = (predictions == labels)
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_accs, bin_confs = [], []
    for i in range(n_bins):
        in_bin = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i+1])
        prop = np.mean(in_bin)
        if prop > 0:
            acc = np.mean(accuracies[in_bin])
            conf = np.mean(confidences[in_bin])
            ece += np.abs(acc - conf) * prop
            bin_accs.append(acc)
            bin_confs.append(conf)
        else:
            bin_accs.append(0.0)
            bin_confs.append((bin_boundaries[i] + bin_boundaries[i+1])/2)
    targets_one_hot = np.zeros_like(probs)
    targets_one_hot[np.arange(len(labels)), labels] = 1.0
    brier = np.mean(np.sum((probs - targets_one_hot)**2, axis=1))
    acc_total = np.mean(accuracies)
    return ece, brier, acc_total, bin_accs, bin_confs

print('--> Đang đánh giá Model 1 Baseline (Cross-Entropy)...')
probs_ce, _ = get_predictions('/content/model_ce_baseline', apply_temp=False)
ece_ce, brier_ce, acc_ce, accs_ce, confs_ce = compute_metrics(probs_ce, true_labels)

print('--> Đang đánh giá Model 1 RLCD Calibrated (Đề xuất)...')
probs_rlcd, temp_rlcd = get_predictions('/content/model_rlcd_calibrated', apply_temp=True)
ece_rlcd, brier_rlcd, acc_rlcd, accs_rlcd, confs_rlcd = compute_metrics(probs_rlcd, true_labels)

# 4. In bảng so sánh đối chứng
headers = ['Mô hình / Phương pháp', 'Nhiệt độ (T)', 'Accuracy (%)', 'ECE (10 Bins) ↓', 'Brier Score ↓']
rows = [
    ['Model 1: Baseline Cross-Entropy', '1.0000 (Mặc định)', f'{acc_ce*100:.2f}%', f'{ece_ce:.4f}', f'{brier_ce:.4f}'],
    ['Model 1: RLCD Calibrated (Đề xuất)', f'{temp_rlcd:.4f} (L-BFGS)', f'{acc_rlcd*100:.2f}%', f'{ece_rlcd:.4f}', f'{brier_rlcd:.4f}']
]
print('\n' + tabulate(rows, headers=headers, tablefmt='fancy_grid'))

# 5. Vẽ biểu đồ Reliability Diagram đối chứng
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot([0, 1], [0, 1], '--', color='gray', label='Perfect Calibration')
plt.plot(confs_ce, accs_ce, 's-', color='tab:red', label=f'Baseline CE (ECE={ece_ce:.3f})')
plt.title('Baseline Cross-Entropy')
plt.xlabel('Confidence')
plt.ylabel('Empirical Accuracy')
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot([0, 1], [0, 1], '--', color='gray', label='Perfect Calibration')
plt.plot(confs_rlcd, accs_rlcd, 'o-', color='tab:blue', label=f'RLCD Calibrated (ECE={ece_rlcd:.3f})')
plt.title(f'RLCD Calibrated (T={temp_rlcd:.2f})')
plt.xlabel('Confidence')
plt.ylabel('Empirical Accuracy')
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plot_out = '/content/reliability_diagram_l4.png'
plt.savefig(plot_out, dpi=200)
plt.show()
print(f' Đã xuất biểu đồ Reliability Diagram tại: {plot_out}')


In [ ]:
# ======================================================================
# BƯỚC 7: XUẤT ONNX FP32 & LƯỢNG TỬ HÓA DYNAMIC INT8 CHO SUY LUẬN CPU
# ======================================================================
!pip install -q onnx onnxruntime onnxscript

import os
import sys
import time
import zipfile
import numpy as np
import torch
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
from onnxruntime.quantization.shape_inference import quant_pre_process
from transformers import AutoTokenizer
from safetensors.torch import load_file

if '/content/kahneman-guard/notebooks/colab_L4' not in sys.path:
    sys.path.insert(0, '/content/kahneman-guard/notebooks/colab_L4')
from train_l4 import KahnemanDecisionGuard

MODEL_DIR = '/content/model_rlcd_calibrated'
ONNX_FP32_PATH = '/content/model_rlcd.onnx'
ONNX_INT8_PATH = '/content/model_rlcd_int8.onnx'
PREP_PATH = '/content/model_rlcd_prep.onnx'

model_id = 'answerdotai/ModernBERT-large'
tok = AutoTokenizer.from_pretrained(model_id)
dummy_text = 'Hello, this is a test prompt for rail system evaluation.'
dummy_enc = tok(dummy_text, max_length=128, padding='max_length', truncation=True, return_tensors='pt')

# [1/4] Kiểm tra nếu đã có file ONNX FP32 từ lần chạy trước thì dùng luôn để tiết kiệm thời gian
if not os.path.exists(ONNX_FP32_PATH) or os.path.getsize(ONNX_FP32_PATH) < 1000:
    print('--> [1/4] Khởi tạo mô hình và xuất sang định dạng ONNX FP32...')
    model = KahnemanDecisionGuard(model_id, use_flash_attn=False)
    weights = load_file(os.path.join(MODEL_DIR, 'model.safetensors'))
    model.load_state_dict(weights)
    model.to(device='cpu', dtype=torch.float32)

    if hasattr(model.encoder, 'gradient_checkpointing_disable'):
        model.encoder.gradient_checkpointing_disable()
    model.eval()

    torch.onnx.export(
        model,
        (dummy_enc['input_ids'], dummy_enc['attention_mask']),
        ONNX_FP32_PATH,
        input_names=['input_ids', 'attention_mask'],
        output_names=['logits'],
        dynamic_axes={'input_ids': {0: 'batch_size', 1: 'seq_len'}, 'attention_mask': {0: 'batch_size', 1: 'seq_len'}},
        opset_version=18,
        do_constant_folding=True
    )
else:
    print(f'--> [1/4] File ONNX FP32 đã tồn tại sẵn ({os.path.getsize(ONNX_FP32_PATH)/(1024*1024):.1f} MB), tiếp tục bước kế tiếp!')

# Xử lý dứt điểm lỗi ONNX Runtime Issue #25890: xóa value_info cũ và chạy quant_pre_process
print('--> Xóa stale value_info và đồng bộ shape metadata...')
m = onnx.load(ONNX_FP32_PATH)
m.graph.ClearField('value_info')
onnx.save(m, ONNX_FP32_PATH)

try:
    quant_pre_process(ONNX_FP32_PATH, PREP_PATH)
    quant_input = PREP_PATH
    print('--> Pre-processing đồ thị ONNX thành công!')
except Exception as e:
    print(f'--> Ghi chú tiền xử lý: {e}')
    quant_input = ONNX_FP32_PATH

print('--> [2/4] Lượng tử hóa Dynamic INT8 tối ưu hóa cho CPU...')
try:
    quantize_dynamic(
        model_input=quant_input,
        model_output=ONNX_INT8_PATH,
        weight_type=QuantType.QInt8,
        extra_options={'DisableShapeInference': True}
    )
except Exception as e:
    print(f'--> Lượng tử hóa MatMul nodes: {e}')
    quantize_dynamic(
        model_input=quant_input,
        model_output=ONNX_INT8_PATH,
        op_types_to_quantize=['MatMul'],
        weight_type=QuantType.QInt8,
        extra_options={'DisableShapeInference': True}
    )

size_int8 = os.path.getsize(ONNX_INT8_PATH) / (1024 * 1024)
print(f'--> Đã xuất ONNX INT8 thành công: {size_int8:.1f} MB!')

print('--> [3/4] Đo lường độ trễ suy luận trên CPU (100 lần thử nghiệm)...')
sess_options = ort.SessionOptions()
sess_options.intra_op_num_threads = 4
session = ort.InferenceSession(ONNX_INT8_PATH, sess_options, providers=['CPUExecutionProvider'])

input_feed = {
    'input_ids': dummy_enc['input_ids'].numpy(),
    'attention_mask': dummy_enc['attention_mask'].numpy()
}
for _ in range(10):
    _ = session.run(None, input_feed)

latencies = []
for _ in range(100):
    t_start = time.perf_counter()
    _ = session.run(None, input_feed)
    latencies.append((time.perf_counter() - t_start) * 1000)

lat_mean = np.mean(latencies)
lat_p95 = np.percentile(latencies, 95)
print(f'\n======================================================')
print(f'  KẾT QUẢ ĐO ĐỘ TRỄ SUY LUẬN CPU (ONNX INT8):')
print(f'  * Độ trễ trung bình:  {lat_mean:.2f} ms')
print(f'  * Phân vị P95:        {lat_p95:.2f} ms')
print(f'  * Tốc độ xử lý:       {1000 / lat_mean:.1f} queries/second')
print(f'======================================================\n')

ZIP_OUT = '/content/kahneman_guard_l4_results.zip'
print(f'--> [4/4] Đang đóng gói toàn bộ Checkpoint & Báo cáo vào {ZIP_OUT}...')
with zipfile.ZipFile(ZIP_OUT, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk('/content/model_rlcd_calibrated'):
        for f in files:
            p = os.path.join(root, f)
            zipf.write(p, os.path.relpath(p, '/content'))
    for root, _, files in os.walk('/content/model_ce_baseline'):
        for f in files:
            p = os.path.join(root, f)
            zipf.write(p, os.path.relpath(p, '/content'))
    for fname in os.listdir('/content'):
        if fname.endswith(('.onnx', '.onnx.data', '.png')):
            p = os.path.join('/content', fname)
            if os.path.isfile(p):
                zipf.write(p, fname)

zip_size_mb = os.path.getsize(ZIP_OUT) / (1024 * 1024)
print(f' HOÀN TẤT TRỌN VẸN! File kết quả đã sẵn sàng: {ZIP_OUT} ({zip_size_mb:.1f} MB)')
print('--> Bạn có thể click chuột phải vào file này ở cột Contents bên trái và chọn Download về máy local!')


# 5. Tự động lưu bản sao dự phòng vào Google Drive (nếu đã kết nối)
try:
    if os.path.exists("/content/drive/MyDrive"):
        import shutil
        shutil.copy(ZIP_OUT, "/content/drive/MyDrive/")
        print("--> [DRIVE] Đã tự động lưu thành công một bản sao vào /content/drive/MyDrive/")
except Exception as e:
    print(f"Lưu ý Drive: {e}")
